In [1]:
import numpy as np
import torch

from math import e

In [2]:
X = np.array([160, 170, 180, 190])


y = np.array([0, 0, 1, 1])

In [3]:
X_mean = np.mean(X)

X_mean

np.float64(175.0)

In [4]:
X_std = np.std(X)

X_std

np.float64(11.180339887498949)

In [5]:
X_norm = (X - X_mean) / X_std

X_norm

array([-1.34164079, -0.4472136 ,  0.4472136 ,  1.34164079])

In [7]:
#파이토치 쓰는 이유 : 미분을 안해도 되는 것.

X_norm_tensor = torch.tensor(X_norm, dtype = torch.float32)
X_norm_tensor

tensor([-1.3416, -0.4472,  0.4472,  1.3416])

In [9]:
y_tensor = torch.tensor(y, dtype = torch.float32)
y_tensor 


tensor([0., 0., 1., 1.])

In [12]:
# 학습 = Cost 값이 0이 되는 지점
a = torch.tensor(0.1, dtype = torch.float32, requires_grad =True)
a

tensor(0.1000, requires_grad=True)

In [ ]:
b = torch.tensor(0.0, dtype = torch.float32, requires_grad = True)
b

tensor(0., requires_grad=True)

In [ ]:
# a의 미분 (z - y - tensor) X_norm_tensor

b의 미분  (z - y_tensor)

In [14]:
def sigmoid(H):
    return 1 / (1+e**(-H))  # 시그모이드 공식

In [17]:
H = a.item()  * X_norm + b.item() #숫자만 보고싶을 떄 : .item()

H

array([-0.13416408, -0.04472136,  0.04472136,  0.13416408])

In [ ]:
sigmoid(H)

array([0.4665092 , 0.48882152, 0.51117848, 0.5334908 ])

In [21]:
z = sigmoid(H)

z

array([0.4665092 , 0.48882152, 0.51117848, 0.5334908 ])

In [23]:
epsilon = 1e-7

z_safe = np.clip(z, epsilon, 1 - epsilon) # 0.99999

z_safe

array([0.4665092 , 0.48882152, 0.51117848, 0.5334908 ])

In [27]:
costs = - ( y * np.log(z_safe ) + (1 - y)*np.log(1-z_safe))

costs

array([0.62831345, 0.67103648, 0.67103648, 0.62831345])

In [29]:
mean_costs = np.mean(costs)

mean_costs

np.float64(0.6496749672265923)

In [30]:
learning_rate = 0.1

epochs = 1000 

In [36]:
for epoch in range(epochs):
    H = a*X_norm_tensor + b
    z = torch.sigmoid(H)
    
    z_safe = torch.clamp(z, epsilon, 1-epsilon)
    
    costs = -(y_tensor * torch.log(z_safe) + (1- y_tensor)*torch.log(1-z_safe))
    mean_cost = torch.mean(costs)
    mean_cost.backward()
    
    grad_a = a.grad.item()
    grad_b = b.grad.item()
    
    with torch.no_grad():
        a -= learning_rate*a.grad
        b -= learning_rate*b.grad
        
    a.grad.zero_()
    b.grad.zero_()
    
    if epoch % 100 == 0 or epoch == epochs - 1:
        print(f"Epoch {epoch}",
              f"cost {mean_cost.item()}",
              f"grad_a {grad_a}",
              f"grad_b {grad_b}",
              f"a {a}",
              f"b {b}"
              )

Epoch 0 cost 0.6320673227310181 grad_a -0.8340030908584595 grad_b 1.4901161193847656e-08 a 0.22562508285045624 b -2.2351742678949904e-09
Epoch 100 cost 0.19604898989200592 grad_a -0.10205328464508057 grad_b 6.51925802230835e-09 a 2.090165853500366 b -1.0244547432591844e-08
Epoch 200 cost 0.13297761976718903 grad_a -0.06258054822683334 grad_b -1.4901161193847656e-08 a 2.8735008239746094 b 1.1102230246251565e-15
Epoch 300 cost 0.10374244302511215 grad_a -0.04689197987318039 grad_b -3.725290298461914e-09 a 3.4111461639404297 b 6.1234479709071366e-09
Epoch 400 cost 0.0858931615948677 grad_a -0.0381079837679863 grad_b 7.334165275096893e-09 a 3.832213878631592 b 3.117602886959503e-08
Epoch 500 cost 0.07357107847929001 grad_a -0.032341450452804565 grad_b -3.725290298461914e-09 a 4.182415962219238 b 3.566384876307893e-08
Epoch 600 cost 0.06444942951202393 grad_a -0.028199918568134308 grad_b -2.9802322387695312e-08 a 4.483890533447266 b 2.7677755198851628e-08
Epoch 700 cost 0.05738283693790436 